In [4]:
# ===== Block 1: Imports =====
import os
import random
import pandas as pd
import numpy as np
from bidict import bidict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import sklearn
import rdkit
from rdkit import Chem

from torch.utils.data import Dataset

from torch_geometric.loader import DataLoader

In [5]:
# ===== Block 2: Settings =====
CACHE_PATH = "../data/graphs_cached.pt"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("====Versions:====")
print("torch:\t", torch.__version__)
print("PyG:\t", torch_geometric.__version__)
print("scikit-learn", sklearn.__version__)
print("device:\t", DEVICE)
if torch.cuda.is_available():
    print("CUDA:\t", torch.version.cuda)

====Versions:====
torch:	 2.10.0+cu128
PyG:	 2.7.0
scikit-learn 1.8.0
device:	 cuda
CUDA:	 12.8


In [6]:
class DonorAcceptorDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df: pd.DataFrame = df.reset_index(drop=True)
        #TODO: define programatically
        self.num_features = 13
        self.num_graph_features = 250
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        donor = row["D_graph"]
        acceptor = row["A_graph"]
        
        return donor, acceptor

In [7]:
from typing import NewType
from rdkit.Chem.rdchem import BondType

AtomicNumber = NewType('AtomicNumber', int)

if os.path.exists(CACHE_PATH):
    cache = torch.load(CACHE_PATH, weights_only=False)
    df: pd.DataFrame = cache['df']
    ATOM_DICT: bidict[AtomicNumber, int] = bidict(cache['atom_vocab'])
    BOND_DICT: bidict[BondType, int] = bidict(cache['bond_vocab'])
    MAX_MOL_SIZE: int = int(cache['max_mol_size'])
else:
    raise(NotImplementedError)

In [ ]:
# mols = pd.concat([df['D_mol'], df['A_mol']] )

In [8]:
n = len(df)
idx = np.arange(n)
np.random.shuffle(idx)

cut = int(0.9 * n)
train_df = df.iloc[idx[:cut]].reset_index(drop=True)
val_df   = df.iloc[idx[cut:]].reset_index(drop=True)

train_ds = DonorAcceptorDataset(train_df)
val_ds   = DonorAcceptorDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

print("Train/Val:", len(train_ds), len(val_ds))

Train/Val: 1597 178


In [9]:
train_ds[0][0]

Data(x=[82, 7], edge_index=[2, 182], edge_attr=[182, 7], graph_attr=[1, 250])

# VGAE
[Bresson and Laurent, 2019](https://ui.adsabs.harvard.edu/link_gateway/2019arXiv190603412B/doi:10.48550/arXiv.1906.03412)


In [ ]:
# ==== Encoder ===== 
from torch_geometric.nn import GATConv

# May redo this to use ConvNet as per the paper
class MolecularEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, graph_in_dim, latent_dim, hidden_dim):
        super().__init__()

        # Broadcast graph features
        in_dim = node_in_dim + graph_in_dim

        self.conv1 = GATConv(in_dim, hidden_dim, edge_dim=edge_in_dim)

        self.conv_mu = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)
        self.conv_logstd = GATConv(hidden_dim, latent_dim, edge_dim=edge_in_dim)

    def forward(self, x, edge_index, edge_attr, graph_attr, batch):
        
        # Broadcast graph attributes
        graph_attr_batch = graph_attr[batch]
        x = torch.cat([x, graph_attr_batch], dim=-1)

        x = F.relu(self.conv1(x, edge_index, edge_attr))

        mu = self.conv_mu(x, edge_index, edge_attr)
        logstd = self.conv_logstd(x, edge_index, edge_attr)

        return mu, logstd

In [ ]:
# ==== Decoder =====
from bidict._bidict import bidict
from torch_geometric.nn.conv.gatv2_conv import GATv2Conv
from torch import Tensor
from torch_geometric.nn import MLP, GATv2Conv


class MolecularDecoder(nn.Module):
    def __init__(self, 
                latent_dim: int, 
                emb_dim: int, 
                gat_hidden_dim=64, 
                max_mol_size: int = MAX_MOL_SIZE,
                atom_dict=ATOM_DICT, 
                edge_dict=BOND_DICT):
        super().__init__()
        self.atom_dict: bidict[AtomicNumber, int] = atom_dict
        self.edge_dict: bidict[BondType, int] = edge_dict
        self.max_size: int = max_mol_size
        self.emb_dim: int = emb_dim

        self.boa_mlp: MLP = MLP(in_channels=latent_dim,
                        out_channels=self.atom_dict_size * self.max_size,
                        hidden_channels=64)

        self.node_emb: nn.Embedding = nn.Embedding(num_embeddings=self.atom_dict_size+self.max_size,
                                                    embedding_dim=self.emb_dim)
        
        self.edge_emb: nn.Linear = nn.Linear(in_features=latent_dim,
                                            out_features=latent_dim,
                                            bias=False)

        # May need to change in_channels
        # TODO: change to take variable number of layers
        self.conv1: GATv2Conv = GATv2Conv(in_channels=emb_dim,
                                        out_channels=gat_hidden_dim,
                                        edge_dim=latent_dim)
        self.conv2: GATv2Conv = GATv2Conv(in_channels=gat_hidden_dim,
                                        out_channels=emb_dim,
                                        edge_dim=latent_dim)

        self.edge_mlp: MLP = MLP(in_channels=emb_dim,
                                out_channels=self.edge_dict_size,
                                hidden_channels=64)

        
    def forward(self, z: Tensor):
        # Atom Generation
        z_soft_boa: Tensor = self.boa_mlp(z).reshape([self.atom_dict_size, self.max_size])
        z_boa = z_soft_boa.max(dim=1).indices

        # Bond Generation
        ## Create a fully connected graph
        num_atoms = int(torch.sum(z_boa))
        atomic_num_idx = torch.arange(self.atom_dict_size)
        edge_index = torch.ones([num_atoms, num_atoms], dtype=torch.bool)

        # Breaking the Symmetry
        # may need to change this to a self.field to keep consistent across the dataset
        max_duplicate_atoms = int(torch.max(z_boa))

        atom_list = torch.repeat_interleave(atomic_num_idx, z_boa).long()

        # Make and concat one-hot vectors for atom type and position (relative to SMILES)
        atom_hots = F.one_hot(atom_list, num_classes=self.atom_dict_size)
        pos_hots = F.one_hot(torch.arange(num_atoms) % max_duplicate_atoms)

        nodes = torch.cat([atom_hots, pos_hots], dim=1)

        # Node embedding
        # Create vector of repeating node_types according to bag of atoms
        # Removed in favor of position+node_type vectors
        # nodes = torch.repeat_interleave(z_boa, z_boa, dim=0)
        x = self.node_emb(nodes)

        ## Edge embedding
        emb_edges: Tensor = self.edge_emb(z)
        edge_attr: Tensor = emb_edges.repeat(edge_index.size())

        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)

        edge_scores = self.edge_mlp(x)

        return x, edge_scores


    #TODO: lookup tables as Tensors? 

    @property
    def atom_dict_size(self) -> int:
        return len(self.atom_dict)
    
    @property
    def edge_dict_size(self) -> int:
        return len(self.edge_dict)
    
    @property
    def max_mol_size(self) -> int:
        return self.max_size

    @property
    def max_edges(self) -> int:
        s = self.max_size
        return s*s

In [ ]:
class MolecularVGAE(nn.Module):
    def __init__(self,
                encoder: MolecularEncoder,
                decoder: MolecularDecoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder